# 🏈 Fantasy Football Draft Analysis (2024–2025)
*Justin McKendry · August 2025*

---

## 📚 Project Overview

This notebook analyzes player performance, draft value, and team efficiency based on our league's draft and season data.  
We explore who maximized their draft capital, which picks over- or under-performed, and how VORP (Value Over Replacement Player) relates to league standings.

**Key Metrics:**
- 📈 VORP (custom-calculated per position)
- 🎯 Draft Delta (Actual Pick - ADP)
- 💥 Boom/Bust Score (Actual - Projected points)

--- 
## 🧪 2. Data Processing

### 🔍 Purpose
Take the data collected in NB01:
- Clean each dataframe
- Filter to input into database
- Create database
- Populate database

# Importing Packages

In [ ]:
import pandas as pd 
import os
import ast
from pathlib import Path
import sqlite3
import re
from rapidfuzz import process, fuzz
from utils import clean_name, read_csvs, generate_schema_from_df, position_from_eligible_slots



# Visualise the distribution of comments per post
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from datetime import datetime
from sqlalchemy import create_engine, text
from config import CURRENT_SEASON, LINEUP_SLOT_MAP, NEXT_SEASON

# Import our custom Reddit API module

# --- Configuration for Jupyter ---
# The following magic command is for Jupyter notebooks to render plots inline.
# It should be commented out when running as a standalone script.
%config InlineBackend.figure_formats = ['svg']


# Get the data from the csvs

In [ ]:
foreign_keys = {
    "player_id": "players(player_id)",
}

next_season_csv = f'../data/raw/{NEXT_SEASON}/projections/espn_proj/{NEXT_SEASON}_proj_stats.csv'
if os.path.exists(next_season_csv):
    next_season_database_df = pd.read_csv(next_season_csv)
else:
    fallback_year = NEXT_SEASON - 1
    print(
        f"No projections CSV yet for {NEXT_SEASON} at {next_season_csv} - run NB01 once "
        f"ESPN publishes {NEXT_SEASON} projections/ADP (usually a few weeks before the draft). "
        f"Falling back to {fallback_year}'s projections, relabeled as {NEXT_SEASON}, so the rest "
        f"of the pipeline (and the live draft tool) still runs end-to-end for local dev/demo."
    )
    next_season_database_df = pd.read_csv(
        f'../data/raw/{fallback_year}/projections/espn_proj/{fallback_year}_proj_stats.csv'
    )
    next_season_database_df['year'] = NEXT_SEASON

next_season_database_df.head()

In [ ]:
# ADP from every season we have a file for, not just 2022+.
#
# 2020 and 2021 were sitting unused in data/raw/ - they use the same
# pre-2026 FantasyPros layout `clean_adp` already handles, and the
# name-based match to `players` further down covers them unchanged
# (2020: 444/580 names matched, 2021: 422/483). Including them takes the
# league-bias fit in src/biases.py from 4 seasons to 6 and from ~683 to
# ~993 usable picks, which is the difference between "suggestive" and
# "solid" for the per-NFL-team effects.

df_adp = read_csvs("adp", years=range(2020, NEXT_SEASON + 1))

df_adp

In [ ]:
df_players = read_csvs("league_stats", "player_stats", range(2020, CURRENT_SEASON + 1))

df_players
cols_to_front = ['player_name', 'player_id','year','team_name','eligible_slots']

# Reorder
df_players = df_players[cols_to_front + [c for c in df_players.columns if c not in cols_to_front]]
df_players = df_players.fillna(0)
df_players[df_players['year'] == 2024]

In [ ]:
df_teams = read_csvs(data="teams_data", years=range(2020, CURRENT_SEASON + 1))


In [ ]:
df_draft = read_csvs(data="draft_data", years=range(2020, CURRENT_SEASON + 1))
df_draft

# Clean Data to Input to SQL Database

### Cleaning df_players

Calculating games_played which is the sum of actual_teamLoss and actual_teamWin. Both those variables are calculations of how their
pro team did WHEN THE PLAYER PLAYED

In [ ]:

df_players['games_played'] = df_players['actual_teamLoss'] + df_players['actual_teamWin']
df_players

### Fixing the `position` column

The `position` column arriving from NB01 was **not** the player's position — it was their
`lineupSlot`, i.e. the roster slot they happened to occupy. That is a weekly roster decision,
not a property of the player, so most rows came through labeled `BE` (bench), `RB/WR/TE`
(started at flex), or a raw, unmapped slot id like `0`:

| value | rows | | value | rows |
|---|---|---|---|---|
| `0` | 1800 | | TE / QB / K / D-ST | 76 each |
| `BE` | 519 | | `RB/WR/TE` | 28 |
| WR / RB | 152 each | | | |

Every downstream notebook filters on `position.isin(POSITIONS)`, so **~76% of every season was
being silently discarded** — and the surviving quarter was biased toward players good enough to
hold a named starting slot, which is exactly the wrong bias for measuring projection accuracy.

`eligible_slots` lists every slot a player is *allowed* to fill, which is a property of the
player, so we derive `position` from that instead (`utils.position_from_eligible_slots`). Matching
is on the singular slot names only — `RB/WR`, `WR/TE`, `RB/WR/TE` and `OP` are ignored, so a WR
eligible at `['RB/WR', 'WR', 'WR/TE', 'RB/WR/TE']` resolves to `WR`, not `RB`.

This is done here, before the ADP cleaning, so that both `players_stats` *and* the
`next_season_projections` position lookup further down get real positions.

In [ ]:
position_before = df_players['position'].value_counts(dropna=False)

df_players['position'] = df_players['eligible_slots'].apply(position_from_eligible_slots)

position_after = df_players['position'].value_counts(dropna=False)
unresolved = df_players['position'].isna().sum()

print("BEFORE (lineup slot, mislabeled as position):")
print(position_before.to_string())
print("\nAFTER (derived from eligible_slots):")
print(position_after.to_string())
print(f"\nUnresolved: {unresolved} of {len(df_players)} rows "
      f"({1 - unresolved / len(df_players):.4%} recovered)")

# Sanity check: the fix must never contradict a label that was already a real
# position. Verified at 100% agreement on the 608 such rows in the current data.
_already_clean = position_before.index.intersection(['QB', 'RB', 'WR', 'TE', 'K', 'D/ST'])
print(f"Positions that were already correct before the fix: {list(_already_clean)}")


### Cleaning df_adp

df_adp contains the average draft position of players across all fantasy football leagues. Our league specifically uses the ESPN app but Sleeper is also used to calculate the average. This can be used as a marker to see if our league draft data is skewed in a certain way.

#### Getting the position rank


Currently this code block extracts the position and the posRank from the POS column. The POS column contains a number like WR2. This means that that player is on average, the second WR to get drafted. It's simpiler for analysis later on if the position and rank are seperate features.

#### Map the Defenses to match the leagues naming convention
In our league, defenses are identified with the team name and D/ST 

i.e.
**Chicago Bears = Bears D/ST** 

I need to map the names to get a consistent formating

#### Match the player names to their league player_id

As adp is pulled from an outside source, their players do not have the same player_id numbers. I have to match the df_adp names to the df_players names to assign them the propper league player_id numbers. Unfortunately both dataframes use different formatting. Suffixes are included in df_adp but not in df_players. I had to remove them from df_adp before matching as well as remove all capitalization just in case. I created a player_clean column in d_adp and matched that to df_players player_clean column. Most players were matched, however, df_adp had more players than df_players so some in df_adp were not assigned a player_id

In [ ]:
# Step 4: Clean names
df_adp["player_clean"] = df_adp["player_name"].apply(clean_name)
df_players["player_clean"] = df_players["player_name"].apply(clean_name)

# Step 5: Create mapping and apply
df_adp = df_adp.merge(
    df_players[["player_id", "player_clean"]],
    how="left",
    on="player_clean"
)

df_players = df_players.drop(columns='player_clean')
df_adp = df_adp.drop(columns='player_clean')



In [ ]:
df_adp['player_id'] = pd.to_numeric(df_adp['player_id'], errors='coerce').astype('Int64')

df_adp = df_adp.drop_duplicates(subset=["year","player_id"], keep="first")
df_adp.fillna(0)

In [ ]:
unmatched_players = df_adp[df_adp["player_id"].isna()]
unmatched_players.to_csv("../data/raw/other/unmatched.csv", index=False)

In [ ]:
df_players[df_players['year'] == 2024] 


#### Converting player_id to Integer

The player_id column in df_adp was a float but in order to merge between tables it needed to be an int.

#### Renaming columns 

Need to rename columns to match the conventions of the database


## Cleaning df_draft

### Map positions in df_draft

This allows us to fill in missing position data. Position data was not included in the df_draft just the lineupSlotId. I had to reverse engineer the positions based off their lineupSlotIds. 

In [ ]:
## RB = 2, WR = 4, WR = 23, QB = 20, TE = 6, K =17, D/ST = 16
df_draft['position'] = df_draft['lineupSlotId'].map(LINEUP_SLOT_MAP)
df_draft.columns

# Filtering Columns in Each Dataframe

Many columns will likely not be used, especially in the df_player. That contains around 100 different features, most of them being null values. 

In [ ]:
current_team_info = (
    df_players[df_players["year"] == CURRENT_SEASON]
    .loc[:, ["player_id", "pro_team", "team_id", "team_name"]]
    .drop_duplicates(subset="player_id")
)
current_team_info = current_team_info.rename(columns={"team_id":"current_team_id", 
                                  "team_name": "current_team_name"})

players_df = (
    df_players
    .loc[:, ["player_id", "player_name"]]
    .drop_duplicates(subset="player_id")
)
players_database_df = players_df.merge(current_team_info, on="player_id", how="left")
stats_database_df = df_players

In [ ]:
#players_database_df = pd.concat(
#    [players_database_df.assign(rookie=0),  # mark vets
#     rookies],
#    ignore_index=True
#)
#players_database_df[players_database_df['rookie'] == 1]

In [ ]:
players_df

In [ ]:
## Extract just the columns needed from the dataframe to prepare to 
## setup the database


adp_columns = [
    'player_name', 
    'team_name', 
    'ESPN', 
    'position',
    'year',
    'Sleeper',
    'POS',
    'pos_rank',
    'avg',  
    'player_id'
]

draft_columns = [
    'autoDraftTypeId', 
    'id', 
    'lineupSlotId',
    'overallPickNumber', 
    'position',
    'player_id', 
    'roundId', 
    'roundPickNumber',
    'team_id',
    'year'
]

### Creating the Final dataframes

These are the final dataframes that will be used in the SQL database

In [ ]:
adp_database_df = df_adp[adp_columns]
draft_database_df = df_draft[draft_columns]
teams_database_df = df_teams

# DataBase Creation

Creating the four different tables.

**draft** = league specific draft data

**players** = professional players and their stats

**teams** = teams within the league

**adp** = worldwide average draft position

### Creating the engine

In [ ]:
engine = create_engine("sqlite:///../data/fantasy_data.db")

### Defining the Schema

In [ ]:
draft_database_df.columns

In [ ]:
draft_database_df

In [ ]:
stats_drop_colums = [
    # All actual_### and proj_### that are just IDs (keep none of them unless mapped)
    'actual_2PtConversions', 'proj_2PtConversions',  # Already covered in touchdowns & points
    'actual_5','actual_6','actual_7','actual_8','actual_9','actual_10','actual_11','actual_12','actual_13','actual_14',
    'proj_5','proj_6','proj_7','proj_8','proj_9','proj_10','proj_11','proj_12',
    'actual_27','actual_28','actual_29','actual_30','actual_31','actual_32','actual_33','actual_34',
    'proj_27','proj_28','proj_29','proj_30','proj_31','proj_33','proj_34',
    'actual_47','actual_48','actual_49','actual_50','actual_51','actual_52','actual_54','actual_55',
    'proj_47','proj_48','proj_49','proj_50','proj_51','proj_54','proj_55',
    'actual_65','proj_65','proj_69',
    'actual_66','actual_67','actual_69','proj_66','proj_67',
    'actual_70','proj_70','proj_71','actual_71',
    'actual_100','proj_100',
    'actual_110','actual_111','actual_112','proj_110','proj_111','proj_112',
    'actual_116','actual_117','proj_116','proj_117',
    'actual_119','proj_119','proj_126','proj_137',
    'actual_143','actual_144','proj_143','proj_144',
    'actual_175','actual_176','actual_177','actual_178',
    'actual_179','actual_180','actual_181','actual_182','proj_198','proj_199','proj_200',
    'actual_183','actual_184','actual_185','proj_210','proj_221','proj_227','proj_233',
    'actual_186','actual_188','actual_189','actual_190','actual_191','actual_192','actual_193','actual_194','actual_195',
    'actual_196','actual_198','actual_199','actual_200',
    'actual_211','actual_212','actual_213','actual_214','actual_215','actual_216','actual_217','actual_218',
    'actual_219','actual_220','actual_221','actual_222','actual_223','actual_224','actual_225','actual_226',
    'actual_227','actual_228','actual_229','actual_230','actual_231','actual_232','actual_233','actual_234',
    'proj_211','proj_212','proj_213','proj_214','proj_215','proj_216','schedule',        # ESPN schedule object — not needed
        # Only needed if you're enforcing lineup logic, not for projections
    'acquisition_type' # Waiver/free agent status, not predictive
]


In [ ]:
stats_database_df = stats_database_df.drop(columns=stats_drop_colums)


In [ ]:
stats_database_df['eligible_slots']

In [ ]:
import json

bad_slots = ['BE', 'IR', 'OP']

def drop_slots(slot_str):
    slot_list = ast.literal_eval(slot_str)
    # Return only the slots that are not in bad_slots
    return [slot for slot in slot_list if slot not in bad_slots]

stats_database_df['eligible_slots'] = stats_database_df['eligible_slots'].apply(drop_slots)
stats_database_df['eligible_slots'] = stats_database_df['eligible_slots']


In [ ]:

stats_database_df["eligible_slots"] = stats_database_df["eligible_slots"].apply(
    lambda v: json.dumps(v if isinstance(v, list) else [])
)
stats_database_df['eligible_slots'].dtype

In [ ]:
foreign_keys = {
    "player_id": "players(player_id)",
    "team_id": "teams(team_id)",
}
players_stats_schema = generate_schema_from_df(
    stats_database_df,
    "players_stats",
    foreign_keys=foreign_keys)

print(players_stats_schema)

In [ ]:
import sys
sys.path.append(os.path.abspath('..'))
from src.scoring import normalize_position
from config import POSITIONS

# next_season_database_df has no `position` column (ESPN's projections export
# doesn't include one) - derive it from the most recently known position for
# each player across our cleaned ADP and player-stats history.
#
# Historically the df_players half of this lookup contributed almost nothing,
# because its `position` column was really `lineupSlot` (see the "Fixing the
# `position` column" section above) and so was filtered out here as invalid.
# Now that df_players carries real positions, it covers players who appear in
# our league's stats but not in the ADP export, which should reduce the
# "no mappable position" count printed below.
#
# The valid-position filter is still applied: ADP occasionally carries labels
# we don't model (e.g. "DT"), and we'd rather drop those than let a garbage
# label win the "most recent" tiebreak against a real one.
valid_positions = set(POSITIONS) | {"D/ST"}
position_lookup = pd.concat([
    df_adp[['player_id', 'year', 'position']],
    df_players[['player_id', 'year', 'position']],
], ignore_index=True).dropna(subset=['player_id', 'position'])
position_lookup['player_id'] = pd.to_numeric(position_lookup['player_id'], errors='coerce')
position_lookup['position'] = position_lookup['position'].map(normalize_position)
position_lookup = position_lookup[position_lookup['position'].isin(valid_positions)]
position_lookup = (
    position_lookup.sort_values('year', ascending=False)
    .drop_duplicates(subset='player_id', keep='first')[['player_id', 'position']]
)

next_season_database_df['player_id'] = pd.to_numeric(next_season_database_df['player_id'], errors='coerce')
next_season_database_df = next_season_database_df.merge(position_lookup, on='player_id', how='left')
print(f"{next_season_database_df['position'].isna().sum()} players with no mappable position "
      f"(e.g. incoming rookies with no prior-season/ADP history)")

In [ ]:
next_season_cols = ['player_id', 'player_name', 'position', 'pro_team', 'projected_points', 'year']
next_season_database_df = next_season_database_df[next_season_cols].dropna(subset=['player_id', 'position'])
next_season_database_df['player_id'] = next_season_database_df['player_id'].astype(int)

next_season_schema = generate_schema_from_df(
    next_season_database_df,
    "next_season_projections",
    foreign_keys=foreign_keys)
print(next_season_schema)
next_season_database_df.head()

In [ ]:
# Define our database schema with specific data types
players_table_schema = """
CREATE TABLE players (
    player_id INTEGER PRIMARY KEY,
    player_name TEXT
);
"""

# player_stats_schema = """
# CREATE TABLE players_stats (
#     player_id CHAR(7),
#     player_name VARCHAR(100),*
#     current_team_name VARCHAR(100),*
#     posRank DECIMAL(4,2),
#     current_team_id VARCHAR(10),*
#     position VARCHAR(20),*
#     pro_team VARCHAR(50),*
#     points DECIMAL(6,2),
#     projected_points DECIMAL(6,2),
#     avg_points DECIMAL(6,2) NOT NULL,
#     projected_avg_points DECIMAL(6,2),
#     actual_pointsScored DECIMAL(7,2),
#     games_played INTEGER,
#     FOREIGN KEY (current_team_id) REFERENCES teams(team_id)
#     FOREIGN KEY(player_id) REFERENCES players(player_id),
#     PRIMARY KEY (player_id, year)
# );
# """

adp_schema = """
CREATE TABLE average_draft_position (
    adp_id INTEGER PRIMARY KEY AUTOINCREMENT,
    player_id CHAR(7),
    player_name VARCHAR(100),
    year INTEGER,
    team_name VARCHAR(50),
    POS VARCHAR(7),
    position STRING,
    pos_rank INTEGER,
    espn INTEGER,
    sleeper INTEGER,
    avg FLOAT,
    FOREIGN KEY (player_id) REFERENCES players(player_id)
);
"""
draft_table_schema = """
CREATE TABLE drafts (
    player_id CHAR(7) PRIMARY KEY,
    year INTEGER,
    position VARCHAR(10),
    overallPickNumber INTEGER,
    team_id VARCHAR(4),
    roundPickNumber INTEGER,
    id INTEGER,
    roundId INTEGER,
    autoDraftTypeId INTEGER,
    lineupSlotId INTEGER,
    FOREIGN KEY (player_id) REFERENCES players(player_id),
    FOREIGN KEY (team_id) REFERENCES teams(team_id)
);
"""

team_table_schema = """
CREATE TABLE teams(
    team_id INTEGER PRIMARY KEY,
    year INTEGER,
    team_name VARCHAR(100),
    abbrev VARCHAR(10),
    division_id INTEGER,
    division_name VARCHAR(100),
    wins INTEGER,
    losses INTEGER,
    ties INTEGER,
    points_for FLOAT,
    points_against FLOAT,
    draft_projected_rank INTEGER,
    final_standing INTEGER
);
"""

# Execute the schema creation
with engine.begin() as conn:   # begin() auto-commits or rolls back
    conn.execute(text("DROP TABLE IF EXISTS drafts;"))
    conn.execute(text("DROP TABLE IF EXISTS average_draft_position;"))
    conn.execute(text("DROP TABLE IF EXISTS players_stats;"))
    conn.execute(text("DROP TABLE IF EXISTS teams;"))
    conn.execute(text("DROP TABLE IF EXISTS players;"))
    conn.execute(text("DROP TABLE IF EXISTS next_season_projections;"))


    conn.execute(text(players_table_schema))
    conn.execute(text(team_table_schema))
    conn.execute(text(players_stats_schema))
    conn.execute(text(adp_schema))
    conn.execute(text(draft_table_schema))
# Database tables created successfully
# - posts table with post_id as PRIMARY KEY
# - comments table with comment_id as PRIMARY KEY and post_id as FOREIGN KEY

### Populating the database

In [ ]:
players_database_df.to_sql('players', engine, if_exists='replace', index=False)
stats_database_df.to_sql('players_stats', engine, if_exists='replace', index=False)
# `append`, not `replace`, on purpose: the schema cell above already dropped and
# recreated this table with `adp_id INTEGER PRIMARY KEY AUTOINCREMENT`, and
# pandas' `replace` would drop it again and rebuild it without that key. The
# flip side is that this cell is NOT safe to re-run on its own - doing so
# appends a second full copy of ADP and silently double-counts every ADP join
# downstream. Re-run the notebook from the schema cell, not from here.
adp_database_df.to_sql('average_draft_position', engine, if_exists='append', index=False)
draft_database_df.to_sql('drafts', engine, if_exists='replace', index=False)
teams_database_df.to_sql('teams', engine, if_exists='replace', index=False)
next_season_database_df.to_sql('next_season_projections', engine, if_exists='replace', index=False)


print(f"✅ Database populated: ADP: {len(adp_database_df)}, Players: {len(players_df)}, Draft: {len(draft_database_df)}")

### Indexes

Created **after** the writes above, never in the schema cell. Every
`to_sql(if_exists="replace")` drops its table *and that table's indexes*, so
anything created earlier would be silently gone by now.

The database was built entirely by `to_sql`, which creates no indexes at all.
The one that matters is the ADP join the live recommender runs on every pick:
measured 57ms to 1.4ms. See `src/indexes.py`.


In [ ]:
from src.indexes import ensure_indexes

created = ensure_indexes(engine, verbose=True)
print(f"✅ {len(created)} indexes present")
